# QwenFree Kaggle server

1. In Notebook options, select **Accelerator → GPU T4 x2** and turn **Internet** on.
2. Add optional Kaggle Secrets named `HF_TOKEN`, `QWENFREE_API_KEY`, and
   `NGROK_AUTHTOKEN`.
3. Review the repository and context values in the next cell, then run all cells.
4. Copy the printed `qwenfree connect` command into Terminal on your Mac.

The final cell runs the authenticated server until you interrupt it. It reports
health every five minutes and does not try to bypass Kaggle session or quota limits.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

def optional_secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for name in ('HF_TOKEN', 'QWENFREE_API_KEY', 'NGROK_AUTHTOKEN'):
    value = optional_secret(name)
    if value:
        os.environ[name] = value

# Change this value to choose another verified Hugging Face GGUF repository.
os.environ.setdefault('MODEL_REPO', 'JonathanColetti/Qwen3.8-27B-Uncensored-GGUF')
os.environ.setdefault('CTX_SIZE', '8192')
os.environ.setdefault('MAX_MODEL_GIB', '24')
print('Secret values were loaded without displaying them.')
print('Selected model repository:', os.environ['MODEL_REPO'])


In [ ]:
from pathlib import Path
import base64

server_dir = Path('/kaggle/working/qwenfree_server')
server_dir.mkdir(parents=True, exist_ok=True)
embedded_files = {
    "bootstrap.sh": "IyEvdXNyL2Jpbi9lbnYgYmFzaApzZXQgLUVldW8gcGlwZWZhaWwKClNDUklQVF9ESVI9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkKSIKV09SS19ESVI9IiR7V09SS19ESVI6LS9rYWdnbGUvd29ya2luZ30iCkxMQU1BX0NQUF9ESVI9IiR7TExBTUFfQ1BQX0RJUjotJHtXT1JLX0RJUn0vbGxhbWEuY3BwfSIKTU9ERUxfRElSPSIke01PREVMX0RJUjotJHtXT1JLX0RJUn0vbW9kZWx9IgpCSU5fRElSPSIke1dPUktfRElSfS9xd2VuZnJlZS1iaW4iCkxPR19ESVI9IiR7V09SS19ESVJ9L3F3ZW5mcmVlLWxvZ3MiClNFUlZFUl9CSU49IiR7TExBTUFfQ1BQX0RJUn0vYnVpbGQvYmluL2xsYW1hLXNlcnZlciIKQ1RYX1NJWkU9IiR7Q1RYX1NJWkU6LTgxOTJ9IgpNQVhfTU9ERUxfR0lCPSIke01BWF9NT0RFTF9HSUI6LTI0fSIKTU9ERUxfUkVQTz0iJHtNT0RFTF9SRVBPOi1Kb25hdGhhbkNvbGV0dGkvUXdlbjMuOC0yN0ItVW5jZW5zb3JlZC1HR1VGfSIKZXhwb3J0IE1PREVMX1JFUE8gTUFYX01PREVMX0dJQgoKU0VSVkVSX1BJRD0iIgpDTE9VREZMQVJFRF9QSUQ9IiIKTkdST0tfUElEPSIiCgpjbGVhbnVwKCkgewogIGxvY2FsIGV4aXRfY29kZT0kPwogIHRyYXAgLSBFWElUIElOVCBURVJNCiAgZWNobyAiU3RvcHBpbmcgUXdlbkZyZWUgc2VydmVyIGFuZCB0dW5uZWwgcHJvY2Vzc2VzLi4uIgogIGZvciBwcm9jZXNzX2lkIGluICIke0NMT1VERkxBUkVEX1BJRH0iICIke05HUk9LX1BJRH0iICIke1NFUlZFUl9QSUR9IjsgZG8KICAgIGlmIFtbIC1uICIke3Byb2Nlc3NfaWR9IiBdXSAmJiBraWxsIC0wICIke3Byb2Nlc3NfaWR9IiAyPi9kZXYvbnVsbDsgdGhlbgogICAgICBraWxsICIke3Byb2Nlc3NfaWR9IiAyPi9kZXYvbnVsbCB8fCB0cnVlCiAgICBmaQogIGRvbmUKICBmb3IgcHJvY2Vzc19pZCBpbiAiJHtDTE9VREZMQVJFRF9QSUR9IiAiJHtOR1JPS19QSUR9IiAiJHtTRVJWRVJfUElEfSI7IGRvCiAgICBpZiBbWyAtbiAiJHtwcm9jZXNzX2lkfSIgXV07IHRoZW4KICAgICAgd2FpdCAiJHtwcm9jZXNzX2lkfSIgMj4vZGV2L251bGwgfHwgdHJ1ZQogICAgZmkKICBkb25lCiAgZXhpdCAiJHtleGl0X2NvZGV9Igp9CnRyYXAgY2xlYW51cCBFWElUCnRyYXAgJ2V4aXQgMTMwJyBJTlQKdHJhcCAnZXhpdCAxNDMnIFRFUk0KCmZvciBkZXBlbmRlbmN5IGluIGdpdCBjbWFrZSBjdXJsIHB5dGhvbjM7IGRvCiAgaWYgISBjb21tYW5kIC12ICIke2RlcGVuZGVuY3l9IiA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIGVjaG8gIlJlcXVpcmVkIGNvbW1hbmQgbm90IGZvdW5kOiAke2RlcGVuZGVuY3l9LiBVc2UgYSBLYWdnbGUgR1BVIG5vdGVib29rLiIgPiYyCiAgICBleGl0IDEKICBmaQpkb25lCgppZiAhIGNvbW1hbmQgLXYgbnZpZGlhLXNtaSA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICBlY2hvICJudmlkaWEtc21pIGlzIG1pc3Npbmc7IGVuYWJsZSBBY2NlbGVyYXRvciAtPiBHUFUgVDQgeDIgaW4gS2FnZ2xlIHNldHRpbmdzLiIgPiYyCiAgZXhpdCAxCmZpCkdQVV9DT1VOVD0iJChudmlkaWEtc21pIC1MIHwgZ3JlcCAtYyAnXkdQVSAnIHx8IHRydWUpIgppZiBbWyAiJHtHUFVfQ09VTlR9IiAhPSAiMiIgXV07IHRoZW4KICBlY2hvICJFeHBlY3RlZCBleGFjdGx5IDIgR1BVcywgZm91bmQgJHtHUFVfQ09VTlR9LiBTZWxlY3QgR1BVIFQ0IHgyIGFuZCByZXJ1bi4iID4mMgogIG52aWRpYS1zbWkgLUwgfHwgdHJ1ZQogIGV4aXQgMQpmaQplY2hvICJEZXRlY3RlZCB0d28gR1BVczoiCm52aWRpYS1zbWkgLS1xdWVyeS1ncHU9bmFtZSxtZW1vcnkudG90YWwgLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyCgppZiBbWyAhICIke0NUWF9TSVpFfSIgPX4gXlswLTldKyQgXV0gfHwgKCggQ1RYX1NJWkUgPCAxMDI0IHx8IENUWF9TSVpFID4gMzI3NjggKSk7IHRoZW4KICBlY2hvICJDVFhfU0laRSBtdXN0IGJlIGFuIGludGVnZXIgZnJvbSAxMDI0IHRvIDMyNzY4OyBuYXRpdmUgMjYyayBjb250ZXh0IGlzIG5vdCB1c2VkLiIgPiYyCiAgZXhpdCAxCmZpCgpta2RpciAtcCAiJHtXT1JLX0RJUn0iICIke0JJTl9ESVJ9IiAiJHtMT0dfRElSfSIgIiR7TU9ERUxfRElSfSIKaWYgISBweXRob24zIC1jICdpbXBvcnQgaHVnZ2luZ2ZhY2VfaHViJyA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICBlY2hvICJJbnN0YWxsaW5nIHRoZSBzbWFsbCBzZXJ2ZXItc2lkZSBIdWdnaW5nIEZhY2UgY2xpZW50Li4uIgogIHB5dGhvbjMgLW0gcGlwIGluc3RhbGwgLS1xdWlldCBodWdnaW5nZmFjZV9odWIKZmkKCmlmIFtbIC1kICIke0xMQU1BX0NQUF9ESVJ9Ly5naXQiIF1dOyB0aGVuCiAgZWNobyAiVXBkYXRpbmcgbGxhbWEuY3BwIGZyb20gaXRzIGNvbmZpZ3VyZWQgZGVmYXVsdCBicmFuY2guLi4iCiAgZ2l0IC1DICIke0xMQU1BX0NQUF9ESVJ9IiBwdWxsIC0tZmYtb25seQplbHNlCiAgZWNobyAiQ2xvbmluZyB0aGUgbGF0ZXN0IGRlZmF1bHQgYnJhbmNoIG9mIGdnbWwtb3JnL2xsYW1hLmNwcC4uLiIKICBnaXQgY2xvbmUgLS1kZXB0aCAxIGh0dHBzOi8vZ2l0aHViLmNvbS9nZ21sLW9yZy9sbGFtYS5jcHAuZ2l0ICIke0xMQU1BX0NQUF9ESVJ9IgpmaQoKZWNobyAiQnVpbGRpbmcgbGxhbWEtc2VydmVyIHdpdGggQ1VEQSBmb3IgTlZJRElBIFQ0IChjb21wdXRlIGNhcGFiaWxpdHkgNy41KS4uLiIKY21ha2UgLVMgIiR7TExBTUFfQ1BQX0RJUn0iIC1CICIke0xMQU1BX0NQUF9ESVJ9L2J1aWxkIiBcCiAgLURHR01MX0NVREE9T04gXAogIC1EQ01BS0VfQ1VEQV9BUkNISVRFQ1RVUkVTPTc1IFwKICAtRENNQUtFX0JVSUxEX1RZUEU9UmVsZWFzZQpjbWFrZSAtLWJ1aWxkICIke0xMQU1BX0NQUF9ESVJ9L2J1aWxkIiAtLWNvbmZpZyBSZWxlYXNlIFwKICAtLXRhcmdldCBsbGFtYS1zZXJ2ZXIgLWogMgoKaWYgW1sgISAteCAiJHtTRVJWRVJfQklOfSIgXV07IHRoZW4KICBlY2hvICJCdWlsZCBjb21wbGV0ZWQsIGJ1dCBsbGFtYS1zZXJ2ZXIgd2FzIG5vdCBmb3VuZCBhdCAke1NFUlZFUl9CSU59LiIgPiYyCiAgZXhpdCAxCmZpCgpTRUxFQ1RJT05fRklMRT0iJHtXT1JLX0RJUn0vcXdlbmZyZWUtbW9kZWwtc2VsZWN0aW9uLmpzb24iCmVjaG8gIkNoZWNraW5nIHRoZSBzZWxlY3RlZCBIdWdnaW5nIEZhY2UgcmVwb3NpdG9yeSBhbmQgR0dVRiBzaXplcy4uLiIKcHl0aG9uMyAiJHtTQ1JJUFRfRElSfS9maW5kX21vZGVsLnB5IiAtLWpzb24gLS1tYXgtbW9kZWwtZ2liICIke01BWF9NT0RFTF9HSUJ9IiBcCiAgPiAiJHtTRUxFQ1RJT05fRklMRX0iCgpweXRob24zIC0gIiR7U0VMRUNUSU9OX0ZJTEV9IiAiJHtNT0RFTF9ESVJ9IiA8PCdQWScKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzeXMKZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IHNuYXBzaG90X2Rvd25sb2FkCgpzZWxlY3Rpb25fcGF0aCwgbW9kZWxfZGlyID0gc3lzLmFyZ3ZbMTpdCndpdGggb3BlbihzZWxlY3Rpb25fcGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgc291cmNlOgogICAgc2VsZWN0aW9uID0ganNvbi5sb2FkKHNvdXJjZSkKcHJpbnQoCiAgICAiRG93bmxvYWRpbmcge30ge30gKHs6LjJmfSBHaUIgb2Ygd2VpZ2h0cykgZnJvbSB7fSIuZm9ybWF0KAogICAgICAgIHNlbGVjdGlvblsicXVhbnQiXSwKICAgICAgICBzZWxlY3Rpb25bInByaW1hcnlfZmlsZSJdLAogICAgICAgIHNlbGVjdGlvblsidG90YWxfc2l6ZV9ieXRlcyJdIC8gMTAyNCoqMywKICAgICAgICBzZWxlY3Rpb25bInJlcG9faWQiXSwKICAgICkKKQpzbmFwc2hvdF9kb3dubG9hZCgKICAgIHJlcG9faWQ9c2VsZWN0aW9uWyJyZXBvX2lkIl0sCiAgICBsb2NhbF9kaXI9bW9kZWxfZGlyLAogICAgYWxsb3dfcGF0dGVybnM9c2VsZWN0aW9uWyJmaWxlcyJdLAogICAgdG9rZW49b3MuZW52aXJvbi5nZXQoIkhGX1RPS0VOIikgb3IgTm9uZSwKKQpmb3IgZmlsZW5hbWUgaW4gc2VsZWN0aW9uWyJmaWxlcyJdOgogICAgaWYgbm90IG9zLnBhdGguaXNmaWxlKG9zLnBhdGguam9pbihtb2RlbF9kaXIsIGZpbGVuYW1lKSk6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiTW9kZWwgZG93bmxvYWQgaXMgaW5jb21wbGV0ZTogIiArIGZpbGVuYW1lKQpQWQoKUFJJTUFSWV9GSUxFPSIkKHB5dGhvbjMgLWMgJ2ltcG9ydCBqc29uLHN5czsgcHJpbnQoanNvbi5sb2FkKG9wZW4oc3lzLmFyZ3ZbMV0pKVsicHJpbWFyeV9maWxlIl0pJyAiJHtTRUxFQ1RJT05fRklMRX0iKSIKTU9ERUxfUEFUSD0iJHtNT0RFTF9ESVJ9LyR7UFJJTUFSWV9GSUxFfSIKaWYgW1sgISAtcyAiJHtNT0RFTF9QQVRIfSIgXV07IHRoZW4KICBlY2hvICJTZWxlY3RlZCBtb2RlbCBmaWxlIGlzIG1pc3Npbmcgb3IgZW1wdHk6ICR7TU9ERUxfUEFUSH0iID4mMgogIGV4aXQgMQpmaQoKU0VSVkVSX0hFTFA9IiQoIiR7U0VSVkVSX0JJTn0iIC0taGVscCAyPiYxIHx8IHRydWUpIgpyZXF1aXJlX3NlcnZlcl9vcHRpb24oKSB7CiAgbG9jYWwgb3B0aW9uPSIkMSIKICBpZiAhIGdyZXAgLUZxIC0tICIke29wdGlvbn0iIDw8PCIke1NFUlZFUl9IRUxQfSI7IHRoZW4KICAgIGVjaG8gIlRoZSBidWlsdCBsbGFtYS1zZXJ2ZXIgZG9lcyBub3Qgc3VwcG9ydCByZXF1aXJlZCBvcHRpb24gJHtvcHRpb259LiIgPiYyCiAgICBlY2hvICJSZWJ1aWxkIGZyb20gdGhlIGxhdGVzdCBsbGFtYS5jcHAgZGVmYXVsdCBicmFuY2ggYW5kIHRyeSBhZ2Fpbi4iID4mMgogICAgZXhpdCAxCiAgZmkKfQpmb3Igb3B0aW9uIGluIC0tdGVuc29yLXNwbGl0IC0tZmxhc2gtYXR0biAtLWN0eC1zaXplIC0taG9zdCAtLXBvcnQgLS1hcGkta2V5IC0tamluamE7IGRvCiAgcmVxdWlyZV9zZXJ2ZXJfb3B0aW9uICIke29wdGlvbn0iCmRvbmUKClNFUlZFUl9BUkdTPSgKICAtbSAiJHtNT0RFTF9QQVRIfSIKICAtbmdsIDk5CiAgLS10ZW5zb3Itc3BsaXQgMSwxCiAgLWMgIiR7Q1RYX1NJWkV9IgogIC0taG9zdCAxMjcuMC4wLjEKICAtLXBvcnQgODA4MAogIC0tamluamEKKQppZiBncmVwIC1GcSAtLSAnLS1mbGFzaC1hdHRuIFtvbnxvZmZ8YXV0b10nIDw8PCIke1NFUlZFUl9IRUxQfSIgfHwgXAogIGdyZXAgLUZxIC0tICctLWZsYXNoLWF0dG4gPG9ufG9mZnxhdXRvPicgPDw8IiR7U0VSVkVSX0hFTFB9IjsgdGhlbgogIFNFUlZFUl9BUkdTKz0oLS1mbGFzaC1hdHRuIG9uKQplbHNlCiAgU0VSVkVSX0FSR1MrPSgtLWZsYXNoLWF0dG4pCmZpCmlmIGdyZXAgLUZxIC0tICctLWNhY2hlLXR5cGUtaycgPDw8IiR7U0VSVkVSX0hFTFB9IiAmJiBcCiAgZ3JlcCAtRnEgLS0gJy0tY2FjaGUtdHlwZS12JyA8PDwiJHtTRVJWRVJfSEVMUH0iICYmIFwKICBncmVwIC1GcSAncThfMCcgPDw8IiR7U0VSVkVSX0hFTFB9IjsgdGhlbgogIFNFUlZFUl9BUkdTKz0oLS1jYWNoZS10eXBlLWsgcThfMCAtLWNhY2hlLXR5cGUtdiBxOF8wKQogIGVjaG8gIlVzaW5nIHE4XzAgS1YgY2FjaGUgYmVjYXVzZSB0aGlzIGJ1aWxkIGFkdmVydGlzZXMgYm90aCBjYWNoZSBmbGFncy4iCmVsc2UKICBlY2hvICJUaGlzIGxsYW1hLXNlcnZlciBidWlsZCBkb2VzIG5vdCBhZHZlcnRpc2UgcThfMCBLViBjYWNoZTsgdXNpbmcgaXRzIGRlZmF1bHQgS1YgY2FjaGUuIgpmaQppZiBncmVwIC1GcSAtLSAnLS1wYXJhbGxlbCcgPDw8IiR7U0VSVkVSX0hFTFB9IjsgdGhlbgogIFNFUlZFUl9BUkdTKz0oLS1wYXJhbGxlbCAxKQpmaQoKaWYgW1sgLW4gIiR7UVdFTkZSRUVfQVBJX0tFWTotfSIgXV07IHRoZW4KICBBUElfS0VZPSIke1FXRU5GUkVFX0FQSV9LRVl9IgplbHNlCiAgQVBJX0tFWT0iJChweXRob24zIC1jICdpbXBvcnQgc2VjcmV0czsgcHJpbnQoc2VjcmV0cy50b2tlbl91cmxzYWZlKDMyKSknKSIKZmkKCmVjaG8gIlN0YXJ0aW5nIGxsYW1hLXNlcnZlciBvbiBsb29wYmFjay4uLiIKIiR7U0VSVkVSX0JJTn0iICIke1NFUlZFUl9BUkdTW0BdfSIgLS1hcGkta2V5ICIke0FQSV9LRVl9IiBcCiAgPiAiJHtMT0dfRElSfS9sbGFtYS1zZXJ2ZXIubG9nIiAyPiYxICYKU0VSVkVSX1BJRD0kIQoKU0VSVkVSX1JFQURZPSJmYWxzZSIKZm9yIF8gaW4gJChzZXEgMSAxODApOyBkbwogIGlmICEga2lsbCAtMCAiJHtTRVJWRVJfUElEfSIgMj4vZGV2L251bGw7IHRoZW4KICAgIGJyZWFrCiAgZmkKICBpZiBjdXJsIC0tc2lsZW50IC0tc2hvdy1lcnJvciAtLWZhaWwgLS1tYXgtdGltZSA1IFwKICAgIGh0dHA6Ly8xMjcuMC4wLjE6ODA4MC9oZWFsdGggPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBTRVJWRVJfUkVBRFk9InRydWUiCiAgICBicmVhawogIGZpCiAgc2xlZXAgNQpkb25lCmlmIFtbICIke1NFUlZFUl9SRUFEWX0iICE9ICJ0cnVlIiBdXTsgdGhlbgogIGlmIGdyZXAgLUVxaSAndW5rbm93biAobW9kZWwgKT9hcmNoaXRlY3R1cmUnICIke0xPR19ESVJ9L2xsYW1hLXNlcnZlci5sb2ciOyB0aGVuCiAgICBlY2hvICJVbmtub3duIG1vZGVsIGFyY2hpdGVjdHVyZTogcmVidWlsZCBsbGFtYS5jcHAgZnJvbSBpdHMgbGF0ZXN0IG1hc3Rlci9kZWZhdWx0IGJyYW5jaC4iID4mMgogIGVsc2UKICAgIGVjaG8gImxsYW1hLXNlcnZlciBkaWQgbm90IGJlY29tZSBoZWFsdGh5LiBSZWNlbnQgc2VydmVyIGxvZyBmb2xsb3dzOiIgPiYyCiAgZmkKICB0YWlsIC1uIDYwICIke0xPR19ESVJ9L2xsYW1hLXNlcnZlci5sb2ciID4mMiB8fCB0cnVlCiAgZXhpdCAxCmZpCgpDTE9VREZMQVJFRF9CSU49IiR7QklOX0RJUn0vY2xvdWRmbGFyZWQiCmlmIFtbICEgLXggIiR7Q0xPVURGTEFSRURfQklOfSIgXV07IHRoZW4KICBlY2hvICJEb3dubG9hZGluZyBjbG91ZGZsYXJlZCBsaW51eC1hbWQ2NCBmcm9tIENsb3VkZmxhcmUncyBvZmZpY2lhbCBHaXRIdWIgcmVsZWFzZS4uLiIKICBjdXJsIC0tZmFpbCAtLWxvY2F0aW9uIC0tc2lsZW50IC0tc2hvdy1lcnJvciBcCiAgICBodHRwczovL2dpdGh1Yi5jb20vY2xvdWRmbGFyZS9jbG91ZGZsYXJlZC9yZWxlYXNlcy9sYXRlc3QvZG93bmxvYWQvY2xvdWRmbGFyZWQtbGludXgtYW1kNjQgXAogICAgLS1vdXRwdXQgIiR7Q0xPVURGTEFSRURfQklOfSIKICBjaG1vZCA3MDAgIiR7Q0xPVURGTEFSRURfQklOfSIKZmkKIiR7Q0xPVURGTEFSRURfQklOfSIgLS12ZXJzaW9uCiIke0NMT1VERkxBUkVEX0JJTn0iIHR1bm5lbCAtLXVybCBodHRwOi8vMTI3LjAuMC4xOjgwODAgXAogID4gIiR7TE9HX0RJUn0vY2xvdWRmbGFyZWQubG9nIiAyPiYxICYKQ0xPVURGTEFSRURfUElEPSQhCgpQVUJMSUNfVVJMPSIiCmZvciBfIGluICQoc2VxIDEgMzApOyBkbwogIFBVQkxJQ19VUkw9IiQoZ3JlcCAtRW8gJ2h0dHBzOi8vW1s6YWxudW06XS1dK1wudHJ5Y2xvdWRmbGFyZVwuY29tJyBcCiAgICAiJHtMT0dfRElSfS9jbG91ZGZsYXJlZC5sb2ciIHwgaGVhZCAtbiAxIHx8IHRydWUpIgogIGlmIFtbIC1uICIke1BVQkxJQ19VUkx9IiBdXTsgdGhlbgogICAgYnJlYWsKICBmaQogIGlmICEga2lsbCAtMCAiJHtDTE9VREZMQVJFRF9QSUR9IiAyPi9kZXYvbnVsbDsgdGhlbgogICAgYnJlYWsKICBmaQogIHNsZWVwIDIKZG9uZQoKaWYgW1sgLXogIiR7UFVCTElDX1VSTH0iICYmIC1uICIke05HUk9LX0FVVEhUT0tFTjotfSIgXV07IHRoZW4KICBlY2hvICJDbG91ZGZsYXJlIHF1aWNrIHR1bm5lbCBkaWQgbm90IHByb3ZpZGUgYSBVUkw7IHRyeWluZyB0aGUgb3B0aW9uYWwgbmdyb2sgZmFsbGJhY2suLi4iCiAgTkdST0tfRElSPSIke0JJTl9ESVJ9L25ncm9rIgogIG1rZGlyIC1wICIke05HUk9LX0RJUn0iCiAgaWYgW1sgISAteCAiJHtOR1JPS19ESVJ9L25ncm9rIiBdXTsgdGhlbgogICAgY3VybCAtLWZhaWwgLS1sb2NhdGlvbiAtLXNpbGVudCAtLXNob3ctZXJyb3IgXAogICAgICBodHRwczovL2Jpbi5uZ3Jvay5jb20vYy9iTnlqMW1RVlk0Yy9uZ3Jvay12My1zdGFibGUtbGludXgtYW1kNjQudGd6IFwKICAgICAgLS1vdXRwdXQgIiR7TkdST0tfRElSfS9uZ3Jvay50Z3oiCiAgICB0YXIgLXh6ZiAiJHtOR1JPS19ESVJ9L25ncm9rLnRneiIgLUMgIiR7TkdST0tfRElSfSIKICAgIGNobW9kIDcwMCAiJHtOR1JPS19ESVJ9L25ncm9rIgogIGZpCiAgTkdST0tfQ09ORklHPSIke1dPUktfRElSfS9xd2VuZnJlZS1uZ3Jvay55bWwiCiAgIiR7TkdST0tfRElSfS9uZ3JvayIgY29uZmlnIGFkZC1hdXRodG9rZW4gIiR7TkdST0tfQVVUSFRPS0VOfSIgXAogICAgLS1jb25maWcgIiR7TkdST0tfQ09ORklHfSIgPi9kZXYvbnVsbAogIGNobW9kIDYwMCAiJHtOR1JPS19DT05GSUd9IgogICIke05HUk9LX0RJUn0vbmdyb2siIGh0dHAgODA4MCAtLWNvbmZpZyAiJHtOR1JPS19DT05GSUd9IiBcCiAgICAtLWxvZyBzdGRvdXQgLS1sb2ctZm9ybWF0IGpzb24gPiAiJHtMT0dfRElSfS9uZ3Jvay5sb2ciIDI+JjEgJgogIE5HUk9LX1BJRD0kIQogIGZvciBfIGluICQoc2VxIDEgMzApOyBkbwogICAgUFVCTElDX1VSTD0iJChweXRob24zIC0gPDwnUFknCmltcG9ydCBqc29uCmltcG9ydCB1cmxsaWIuZXJyb3IKaW1wb3J0IHVybGxpYi5yZXF1ZXN0Cgp0cnk6CiAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4oImh0dHA6Ly8xMjcuMC4wLjE6NDA0MC9hcGkvdHVubmVscyIsIHRpbWVvdXQ9MSkgYXMgcmVzcG9uc2U6CiAgICAgICAgdHVubmVscyA9IGpzb24ubG9hZChyZXNwb25zZSkuZ2V0KCJ0dW5uZWxzIiwgW10pCiAgICBwcmludChuZXh0KChpdGVtWyJwdWJsaWNfdXJsIl0gZm9yIGl0ZW0gaW4gdHVubmVscyBpZiBpdGVtLmdldCgicHJvdG8iKSA9PSAiaHR0cHMiKSwgIiIpKQpleGNlcHQgKE9TRXJyb3IsIHVybGxpYi5lcnJvci5VUkxFcnJvciwganNvbi5KU09ORGVjb2RlRXJyb3IpOgogICAgcHJpbnQoIiIpClBZCiAgICApIgogICAgaWYgW1sgLW4gIiR7UFVCTElDX1VSTH0iIF1dOyB0aGVuCiAgICAgIGJyZWFrCiAgICBmaQogICAgc2xlZXAgMgogIGRvbmUKZmkKCmlmIFtbIC16ICIke1BVQkxJQ19VUkx9IiBdXTsgdGhlbgogIGVjaG8gIk5vIHB1YmxpYyB0dW5uZWwgVVJMIHdhcyBvYnRhaW5lZC4gUmVjZW50IHR1bm5lbCBsb2cgZm9sbG93czoiID4mMgogIHRhaWwgLW4gNDAgIiR7TE9HX0RJUn0vY2xvdWRmbGFyZWQubG9nIiA+JjIgfHwgdHJ1ZQogIGlmIFtbIC16ICIke05HUk9LX0FVVEhUT0tFTjotfSIgXV07IHRoZW4KICAgIGVjaG8gIlNldCB0aGUgb3B0aW9uYWwgTkdST0tfQVVUSFRPS0VOIEthZ2dsZSBTZWNyZXQgdG8gZW5hYmxlIG5ncm9rIGZhbGxiYWNrLiIgPiYyCiAgZmkKICBleGl0IDEKZmkKCmNhdCA8PEVPRgoKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09ClF3ZW5GcmVlIHNlcnZlciBpcyByZWFkeS4gS2VlcCB0aGlzIEthZ2dsZSBjZWxsIHJ1bm5pbmcuClFXRU5GUkVFX1VSTD0ke1BVQkxJQ19VUkx9ClFXRU5GUkVFX0tFWT0ke0FQSV9LRVl9CgpPbiB5b3VyIE1hYywgcnVuOgpxd2VuZnJlZSBjb25uZWN0ICR7UFVCTElDX1VSTH0gLS1rZXkgJHtBUElfS0VZfQo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KRU9GCgp3aGlsZSBraWxsIC0wICIke1NFUlZFUl9QSUR9IiAyPi9kZXYvbnVsbDsgZG8KICBzbGVlcCAzMDAKICBpZiBjdXJsIC0tc2lsZW50IC0tc2hvdy1lcnJvciAtLWZhaWwgLS1tYXgtdGltZSAxMCBcCiAgICBodHRwOi8vMTI3LjAuMC4xOjgwODAvaGVhbHRoID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAgZWNobyAiJChkYXRlIC11ICcrJVktJW0tJWQgJUg6JU06JVMgVVRDJykgUXdlbkZyZWUgc2VydmVyIGhlYWx0aDogT0siCiAgZWxzZQogICAgZWNobyAiJChkYXRlIC11ICcrJVktJW0tJWQgJUg6JU06JVMgVVRDJykgUXdlbkZyZWUgc2VydmVyIGhlYWx0aDogdW5hdmFpbGFibGUiCiAgZmkKZG9uZQoKZWNobyAibGxhbWEtc2VydmVyIHN0b3BwZWQ7IGNoZWNrICR7TE9HX0RJUn0vbGxhbWEtc2VydmVyLmxvZy4iCg==",
    "find_model.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTZWxlY3QgYSB2ZXJpZmllZCBRd2VuMy44LTI3QiB1bmNlbnNvcmVkIEdHVUYgZnJvbSBIdWdnaW5nIEZhY2UuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3lzCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdCwgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUHVyZVBvc2l4UGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpERUZBVUxUX1JFUE8gPSAiSm9uYXRoYW5Db2xldHRpL1F3ZW4zLjgtMjdCLVVuY2Vuc29yZWQtR0dVRiIKREVGQVVMVF9NQVhfTU9ERUxfR0lCID0gMjQuMApRVUFOVF9PUkRFUiA9ICgiUTRfS19NIiwgIklRNF9YUyIsICJRM19LX00iKQpHSUIgPSAxMDI0KiozCgoKY2xhc3MgTW9kZWxTZWxlY3Rpb25FcnJvcihSdW50aW1lRXJyb3IpOgogICAgIiIiUmFpc2VkIHdoZW4gbm8gdmVyaWZpZWQgR0dVRiBmaXRzIHRoZSBjb25maWd1cmVkIG1lbW9yeSBidWRnZXQuIiIiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgTW9kZWxTZWxlY3Rpb246CiAgICAiIiJUaGUgZXhhY3QgZmlsZXMgYW5kIG1vZGVsIG1ldGFkYXRhIG5lZWRlZCBieSB0aGUgYm9vdHN0cmFwIHNjcmlwdC4iIiIKCiAgICByZXBvX2lkOiBzdHIKICAgIHF1YW50OiBzdHIKICAgIGZpbGVzOiBsaXN0W3N0cl0KICAgIHByaW1hcnlfZmlsZTogc3RyCiAgICB0b3RhbF9zaXplX2J5dGVzOiBpbnQKICAgIG1heF9tb2RlbF9naWI6IGZsb2F0CgoKZGVmIF9nZXQoaXRlbTogQW55LCBrZXk6IHN0ciwgZGVmYXVsdDogQW55ID0gTm9uZSkgLT4gQW55OgogICAgIiIiUmVhZCBhIGZpZWxkIGZyb20gZWl0aGVyIEhGIG1ldGFkYXRhIG9iamVjdHMgb3IgdGVzdCBkaWN0aW9uYXJpZXMuIiIiCiAgICBpZiBpc2luc3RhbmNlKGl0ZW0sIGRpY3QpOgogICAgICAgIHJldHVybiBpdGVtLmdldChrZXksIGRlZmF1bHQpCiAgICByZXR1cm4gZ2V0YXR0cihpdGVtLCBrZXksIGRlZmF1bHQpCgoKZGVmIHF1YW50X2Zvcl9maWxlbmFtZShmaWxlbmFtZTogc3RyKSAtPiBzdHIgfCBOb25lOgogICAgIiIiUmV0dXJuIGEgc3VwcG9ydGVkIHF1YW50IG5hbWUgd2hlbiBwcmVzZW50IGluIGEgR0dVRiBmaWxlbmFtZS4iIiIKICAgIGlmIG5vdCBmaWxlbmFtZS5sb3dlcigpLmVuZHN3aXRoKCIuZ2d1ZiIpOgogICAgICAgIHJldHVybiBOb25lCiAgICBsb3dlcmVkID0gZmlsZW5hbWUubG93ZXIoKQogICAgcGF0dGVybnMgPSB7CiAgICAgICAgIlE0X0tfTSI6IHIiKD88IVthLXowLTldKXE0W18tXT9rW18tXT9tKD8hW2EtejAtOV0pIiwKICAgICAgICAiSVE0X1hTIjogciIoPzwhW2EtejAtOV0paXE0W18tXT94cyg/IVthLXowLTldKSIsCiAgICAgICAgIlEzX0tfTSI6IHIiKD88IVthLXowLTldKXEzW18tXT9rW18tXT9tKD8hW2EtejAtOV0pIiwKICAgIH0KICAgIGZvciBxdWFudCwgcGF0dGVybiBpbiBwYXR0ZXJucy5pdGVtcygpOgogICAgICAgIGlmIHJlLnNlYXJjaChwYXR0ZXJuLCBsb3dlcmVkKToKICAgICAgICAgICAgcmV0dXJuIHF1YW50CiAgICByZXR1cm4gTm9uZQoKCmRlZiBfc2FmZV9yZXBvX2ZpbGVuYW1lKGZpbGVuYW1lOiBzdHIpIC0+IGJvb2w6CiAgICAiIiJSZWplY3QgYWJzb2x1dGUgcGF0aHMgYW5kIHRyYXZlcnNhbCBpbiB1bnRydXN0ZWQgSHViIGZpbGUgbWV0YWRhdGEuIiIiCiAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChmaWxlbmFtZSkKICAgIHJldHVybiBib29sKGZpbGVuYW1lKSBhbmQgbm90IHBhdGguaXNfYWJzb2x1dGUoKSBhbmQgIlxcIiBub3QgaW4gZmlsZW5hbWUgYW5kIGFsbCgKICAgICAgICBwYXJ0IG5vdCBpbiB7IiIsICIuIiwgIi4uIn0gZm9yIHBhcnQgaW4gcGF0aC5wYXJ0cwogICAgKQoKCmRlZiBfZ3JvdXBzX2Zvcl9xdWFudChzaWJsaW5nczogbGlzdFtBbnldLCBxdWFudDogc3RyKSAtPiBsaXN0W3R1cGxlW2xpc3Rbc3RyXSwgaW50XV06CiAgICAiIiJHcm91cCBhIHF1YW50J3MgZmlsZXMgaW50byBzaW5nbGUgR0dVRnMgb3IgY29tcGxldGUgc2hhcmQgc2V0cy4iIiIKICAgIGdyb3VwczogZGljdFtzdHIsIGRpY3Rbc3RyLCBBbnldXSA9IHt9CiAgICBmb3Igc2libGluZyBpbiBzaWJsaW5nczoKICAgICAgICBmaWxlbmFtZSA9IHN0cihfZ2V0KHNpYmxpbmcsICJyZmlsZW5hbWUiLCAiIikpCiAgICAgICAgc2l6ZSA9IF9nZXQoc2libGluZywgInNpemUiKQogICAgICAgIGlmICgKICAgICAgICAgICAgbm90IF9zYWZlX3JlcG9fZmlsZW5hbWUoZmlsZW5hbWUpCiAgICAgICAgICAgIG9yIHF1YW50X2Zvcl9maWxlbmFtZShmaWxlbmFtZSkgIT0gcXVhbnQKICAgICAgICAgICAgb3Igbm90IGlzaW5zdGFuY2Uoc2l6ZSwgaW50KQogICAgICAgICAgICBvciBzaXplIDw9IDAKICAgICAgICApOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHNwbGl0ID0gcmUubWF0Y2goCiAgICAgICAgICAgIHIiXig/UDxzdGVtPi4qKS0oP1A8aW5kZXg+XGR7NX0pLW9mLSg/UDxjb3VudD5cZHs1fSlcLmdndWYkIiwKICAgICAgICAgICAgZmlsZW5hbWUsCiAgICAgICAgICAgIGZsYWdzPXJlLklHTk9SRUNBU0UsCiAgICAgICAgKQogICAgICAgIGlmIHNwbGl0OgogICAgICAgICAgICBzdGVtID0gc3BsaXQuZ3JvdXAoInN0ZW0iKS5sb3dlcigpCiAgICAgICAgICAgIGdyb3VwID0gZ3JvdXBzLnNldGRlZmF1bHQoCiAgICAgICAgICAgICAgICBzdGVtLAogICAgICAgICAgICAgICAgeyJmaWxlcyI6IFtdLCAiY291bnQiOiBpbnQoc3BsaXQuZ3JvdXAoImNvdW50IikpLCAiaW5kaWNlcyI6IFtdfSwKICAgICAgICAgICAgKQogICAgICAgICAgICBpZiBncm91cFsiY291bnQiXSAhPSBpbnQoc3BsaXQuZ3JvdXAoImNvdW50IikpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZ3JvdXBbImZpbGVzIl0uYXBwZW5kKChpbnQoc3BsaXQuZ3JvdXAoImluZGV4IikpLCBmaWxlbmFtZSwgc2l6ZSkpCiAgICAgICAgICAgIGdyb3VwWyJpbmRpY2VzIl0uYXBwZW5kKGludChzcGxpdC5ncm91cCgiaW5kZXgiKSkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZ3JvdXBzW2ZpbGVuYW1lLmxvd2VyKCldID0gewogICAgICAgICAgICAgICAgImZpbGVzIjogWygxLCBmaWxlbmFtZSwgc2l6ZSldLAogICAgICAgICAgICAgICAgImNvdW50IjogMSwKICAgICAgICAgICAgICAgICJpbmRpY2VzIjogWzFdLAogICAgICAgICAgICB9CgogICAgdmFsaWQ6IGxpc3RbdHVwbGVbbGlzdFtzdHJdLCBpbnRdXSA9IFtdCiAgICBmb3IgZ3JvdXAgaW4gZ3JvdXBzLnZhbHVlcygpOgogICAgICAgIGV4cGVjdGVkID0gZ3JvdXBbImNvdW50Il0KICAgICAgICBmaWxlc193aXRoX3NpemUgPSBncm91cFsiZmlsZXMiXQogICAgICAgIGluZGljZXMgPSBncm91cFsiaW5kaWNlcyJdCiAgICAgICAgaWYgbGVuKGZpbGVzX3dpdGhfc2l6ZSkgIT0gZXhwZWN0ZWQgb3Igc29ydGVkKGluZGljZXMpICE9IGxpc3QoCiAgICAgICAgICAgIHJhbmdlKDEsIGV4cGVjdGVkICsgMSkKICAgICAgICApOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZpbGVzX3dpdGhfc2l6ZS5zb3J0KGtleT1sYW1iZGEgaXRlbTogaXRlbVswXSkKICAgICAgICBmaWxlbmFtZXMgPSBbaXRlbVsxXSBmb3IgaXRlbSBpbiBmaWxlc193aXRoX3NpemVdCiAgICAgICAgc2l6ZSA9IHN1bShpdGVtWzJdIGZvciBpdGVtIGluIGZpbGVzX3dpdGhfc2l6ZSkKICAgICAgICB2YWxpZC5hcHBlbmQoKGZpbGVuYW1lcywgc2l6ZSkpCiAgICByZXR1cm4gdmFsaWQKCgpkZWYgc2VsZWN0X3F1YW50KAogICAgcmVwb19pZDogc3RyLAogICAgc2libGluZ3M6IGxpc3RbQW55XSwKICAgICosCiAgICBtYXhfbW9kZWxfZ2liOiBmbG9hdCA9IERFRkFVTFRfTUFYX01PREVMX0dJQiwKKSAtPiBNb2RlbFNlbGVjdGlvbjoKICAgICIiIlBpY2sgdGhlIHByZWZlcnJlZCBjb21wbGV0ZSBxdWFudCB0aGF0IGZpdHMgdW5kZXIgdGhlIHdlaWdodCBidWRnZXQuIiIiCiAgICBpZiBtYXhfbW9kZWxfZ2liIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWF4X21vZGVsX2dpYiBtdXN0IGJlIHBvc2l0aXZlLiIpCiAgICBtYXhfYnl0ZXMgPSBpbnQobWF4X21vZGVsX2dpYiAqIEdJQikKICAgIGZvciBxdWFudCBpbiBRVUFOVF9PUkRFUjoKICAgICAgICBjYW5kaWRhdGVzID0gWwogICAgICAgICAgICBncm91cAogICAgICAgICAgICBmb3IgZ3JvdXAgaW4gX2dyb3Vwc19mb3JfcXVhbnQoc2libGluZ3MsIHF1YW50KQogICAgICAgICAgICBpZiBncm91cFsxXSA8PSBtYXhfYnl0ZXMKICAgICAgICBdCiAgICAgICAgaWYgY2FuZGlkYXRlczoKICAgICAgICAgICAgY2FuZGlkYXRlcy5zb3J0KAogICAgICAgICAgICAgICAga2V5PWxhbWJkYSBncm91cDogKAogICAgICAgICAgICAgICAgICAgIGFueSgibm9tdHAiIGluIG5hbWUubG93ZXIoKSBmb3IgbmFtZSBpbiBncm91cFswXSksCiAgICAgICAgICAgICAgICAgICAgZ3JvdXBbMV0sCiAgICAgICAgICAgICAgICAgICAgZ3JvdXBbMF1bMF0ubG93ZXIoKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAgICAgICBmaWxlbmFtZXMsIHRvdGFsX3NpemUgPSBjYW5kaWRhdGVzWzBdCiAgICAgICAgICAgIHJldHVybiBNb2RlbFNlbGVjdGlvbigKICAgICAgICAgICAgICAgIHJlcG9faWQ9cmVwb19pZCwKICAgICAgICAgICAgICAgIHF1YW50PXF1YW50LAogICAgICAgICAgICAgICAgZmlsZXM9ZmlsZW5hbWVzLAogICAgICAgICAgICAgICAgcHJpbWFyeV9maWxlPWZpbGVuYW1lc1swXSwKICAgICAgICAgICAgICAgIHRvdGFsX3NpemVfYnl0ZXM9dG90YWxfc2l6ZSwKICAgICAgICAgICAgICAgIG1heF9tb2RlbF9naWI9bWF4X21vZGVsX2dpYiwKICAgICAgICAgICAgKQogICAgcmFpc2UgTW9kZWxTZWxlY3Rpb25FcnJvcigKICAgICAgICAiTm8gY29tcGxldGUgUTRfS19NLCBJUTRfWFMsIG9yIFEzX0tfTSBHR1VGIGZpdHMgdGhlICIKICAgICAgICBmInttYXhfbW9kZWxfZ2liOmd9IEdpQiB3ZWlnaHQgYnVkZ2V0IGluIHtyZXBvX2lkfS4iCiAgICApCgoKZGVmIF9oYXNfZ2d1Zl9maWxlcyhhcGk6IEFueSwgcmVwb19pZDogc3RyLCB0b2tlbjogc3RyIHwgTm9uZSkgLT4gYm9vbDoKICAgICIiIkNoZWNrIGEgc2VhcmNoIHJlc3VsdCBmb3IgR0dVRiBmaWxlcyB3aXRob3V0IGRvd25sb2FkaW5nIG1vZGVsIGRhdGEuIiIiCiAgICB0cnk6CiAgICAgICAgaW5mbyA9IGFwaS5tb2RlbF9pbmZvKHJlcG9faWQsIGZpbGVzX21ldGFkYXRhPUZhbHNlLCB0b2tlbj10b2tlbikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICByZXR1cm4gYW55KAogICAgICAgIHN0cihfZ2V0KGl0ZW0sICJyZmlsZW5hbWUiLCAiIikpLmxvd2VyKCkuZW5kc3dpdGgoIi5nZ3VmIikKICAgICAgICBmb3IgaXRlbSBpbiAoX2dldChpbmZvLCAic2libGluZ3MiLCBbXSkgb3IgW10pCiAgICApCgoKZGVmIGRpc2NvdmVyX2NhbmRpZGF0ZV9yZXBvcyhhcGk6IEFueSwgdG9rZW46IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBsaXN0W3N0cl06CiAgICAiIiJGaW5kIG90aGVyIHNpbWlsYXJseSBuYW1lZCB1bmNlbnNvcmVkIEdHVUYgcmVwb3NpdG9yaWVzIGZvciB1c2VyIGNob2ljZS4iIiIKICAgIHRyeToKICAgICAgICBtb2RlbHMgPSBhcGkubGlzdF9tb2RlbHMoc2VhcmNoPSJRd2VuMy44LTI3QiIsIGxpbWl0PTEwMCwgdG9rZW49dG9rZW4pCiAgICBleGNlcHQgVHlwZUVycm9yOgogICAgICAgIG1vZGVscyA9IGFwaS5saXN0X21vZGVscyhzZWFyY2g9IlF3ZW4zLjgtMjdCIikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIFtdCgogICAgY2FuZGlkYXRlczogbGlzdFtzdHJdID0gW10KICAgIGN1ZXMgPSAoInVuY2Vuc29yZWQiLCAidW5sZWFzaGVkIiwgImFibGl0ZXJhdCIsICJoZXJldGljIikKICAgIGZvciBtb2RlbCBpbiBtb2RlbHM6CiAgICAgICAgcmVwb19pZCA9IHN0cihfZ2V0KG1vZGVsLCAiaWQiLCAiIikgb3IgX2dldChtb2RlbCwgIm1vZGVsSWQiLCAiIikpCiAgICAgICAgbG93ZXJlZCA9IHJlcG9faWQubG93ZXIoKQogICAgICAgIGlmIG5vdCByZXBvX2lkIG9yIG5vdCBhbnkoY3VlIGluIGxvd2VyZWQgZm9yIGN1ZSBpbiBjdWVzKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBfaGFzX2dndWZfZmlsZXMoYXBpLCByZXBvX2lkLCB0b2tlbik6CiAgICAgICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKHJlcG9faWQpCiAgICAgICAgaWYgbGVuKGNhbmRpZGF0ZXMpID49IDEyOgogICAgICAgICAgICBicmVhawogICAgcmV0dXJuIHNvcnRlZChzZXQoY2FuZGlkYXRlcyksIGtleT1zdHIubG93ZXIpCgoKZGVmIHJlc29sdmVfbW9kZWwoCiAgICBhcGk6IEFueSwKICAgIHJlcG9faWQ6IHN0ciwKICAgICosCiAgICB0b2tlbjogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICBtYXhfbW9kZWxfZ2liOiBmbG9hdCA9IERFRkFVTFRfTUFYX01PREVMX0dJQiwKICAgIG1pc3NpbmdfcmVwb19lcnJvcnM6IHR1cGxlW3R5cGVbQmFzZUV4Y2VwdGlvbl0sIC4uLl0gPSAoKSwKKSAtPiBNb2RlbFNlbGVjdGlvbjoKICAgICIiIlZlcmlmeSBhIHJlcG9zaXRvcnksIG9yIHNob3cgc2VhcmNoIHJlc3VsdHMgaWYgaXQgaXMgbWlzc2luZy4iIiIKICAgIHRyeToKICAgICAgICBpbmZvID0gYXBpLm1vZGVsX2luZm8ocmVwb19pZCwgZmlsZXNfbWV0YWRhdGE9VHJ1ZSwgdG9rZW49dG9rZW4pCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBpZiBub3QgbWlzc2luZ19yZXBvX2Vycm9ycyBvciBub3QgaXNpbnN0YW5jZShleGMsIG1pc3NpbmdfcmVwb19lcnJvcnMpOgogICAgICAgICAgICByYWlzZQogICAgICAgIGNhbmRpZGF0ZXMgPSBkaXNjb3Zlcl9jYW5kaWRhdGVfcmVwb3MoYXBpLCB0b2tlbikKICAgICAgICBzdWZmaXggPSAiXG4iLmpvaW4oZiIgIHtjYW5kaWRhdGV9IiBmb3IgY2FuZGlkYXRlIGluIGNhbmRpZGF0ZXMpCiAgICAgICAgZ3VpZGFuY2UgPSAoCiAgICAgICAgICAgICJTZXQgTU9ERUxfUkVQTyBpbiB0aGUgS2FnZ2xlIG5vdGVib29rIHRvIG9uZSBvZiB0aGVzZSByZXBvc2l0b3JpZXMsICIKICAgICAgICAgICAgInRoZW4gcmVydW4gdGhlIHNlcnZlciBjZWxsLiIKICAgICAgICApCiAgICAgICAgaWYgc3VmZml4OgogICAgICAgICAgICBndWlkYW5jZSA9IGYie2d1aWRhbmNlfVxuQ2FuZGlkYXRlczpcbntzdWZmaXh9IgogICAgICAgIHJhaXNlIE1vZGVsU2VsZWN0aW9uRXJyb3IoCiAgICAgICAgICAgIGYiUmVxdWVzdGVkIEh1Z2dpbmcgRmFjZSByZXBvc2l0b3J5IHdhcyBub3QgZm91bmQ6IHtyZXBvX2lkfS4ge2d1aWRhbmNlfSIKICAgICAgICApIGZyb20gZXhjCgogICAgc2libGluZ3MgPSBsaXN0KF9nZXQoaW5mbywgInNpYmxpbmdzIiwgW10pIG9yIFtdKQogICAgaWYgbm90IHNpYmxpbmdzOgogICAgICAgIHJhaXNlIE1vZGVsU2VsZWN0aW9uRXJyb3IoZiJSZXBvc2l0b3J5IHtyZXBvX2lkfSBjb250YWlucyBubyBmaWxlcy4iKQogICAgcmV0dXJuIHNlbGVjdF9xdWFudChyZXBvX2lkLCBzaWJsaW5ncywgbWF4X21vZGVsX2dpYj1tYXhfbW9kZWxfZ2liKQoKCmRlZiBtYWluKGFyZ3Y6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lKSAtPiBpbnQ6CiAgICAiIiJSZXNvbHZlIGEgcmVwb3NpdG9yeSB0aHJvdWdoIHRoZSBIdWdnaW5nIEZhY2UgQVBJIGFuZCBwcmludCBpdHMgY2hvaWNlLiIiIgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVwbyIsIGRlZmF1bHQ9b3MuZW52aXJvbi5nZXQoIk1PREVMX1JFUE8iLCBERUZBVUxUX1JFUE8pKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tYXgtbW9kZWwtZ2liIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD1ERUZBVUxUX01BWF9NT0RFTF9HSUIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLWpzb24iLCBhY3Rpb249InN0b3JlX3RydWUiLCBoZWxwPSJQcmludCBtYWNoaW5lLXJlYWRhYmxlIEpTT04uIgogICAgKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKGFyZ3YpCgogICAgdHJ5OgogICAgICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBIZkFwaQogICAgICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViLnV0aWxzIGltcG9ydCBSZXBvc2l0b3J5Tm90Rm91bmRFcnJvcgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHByaW50KAogICAgICAgICAgICAiaHVnZ2luZ2ZhY2VfaHViIGlzIHJlcXVpcmVkIGluIHRoZSBLYWdnbGUgc2VydmVyIGVudmlyb25tZW50LiIsCiAgICAgICAgICAgIGZpbGU9c3lzLnN0ZGVyciwKICAgICAgICApCiAgICAgICAgcmV0dXJuIDIKCiAgICB0b2tlbiA9IG9zLmVudmlyb24uZ2V0KCJIRl9UT0tFTiIpIG9yIE5vbmUKICAgIHRyeToKICAgICAgICBzZWxlY3Rpb24gPSByZXNvbHZlX21vZGVsKAogICAgICAgICAgICBIZkFwaSgpLAogICAgICAgICAgICBhcmdzLnJlcG8sCiAgICAgICAgICAgIHRva2VuPXRva2VuLAogICAgICAgICAgICBtYXhfbW9kZWxfZ2liPWFyZ3MubWF4X21vZGVsX2dpYiwKICAgICAgICAgICAgbWlzc2luZ19yZXBvX2Vycm9ycz0oUmVwb3NpdG9yeU5vdEZvdW5kRXJyb3IsKSwKICAgICAgICApCiAgICBleGNlcHQgTW9kZWxTZWxlY3Rpb25FcnJvciBhcyBleGM6CiAgICAgICAgcHJpbnQoc3RyKGV4YyksIGZpbGU9c3lzLnN0ZGVycikKICAgICAgICByZXR1cm4gMgogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiSHVnZ2luZyBGYWNlIGxvb2t1cCBmYWlsZWQgKHt0eXBlKGV4YykuX19uYW1lX199KTsgY2hlY2sgS2FnZ2xlIEludGVybmV0ICIKICAgICAgICAgICAgImFjY2VzcyBhbmQgcmVwb3NpdG9yeSBwZXJtaXNzaW9ucy4iLAogICAgICAgICAgICBmaWxlPXN5cy5zdGRlcnIsCiAgICAgICAgKQogICAgICAgIHJldHVybiAxCgogICAgcmVzdWx0ID0gYXNkaWN0KHNlbGVjdGlvbikKICAgIGlmIGFyZ3MuanNvbjoKICAgICAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdCkpCiAgICBlbHNlOgogICAgICAgIHByaW50KGYiUmVwb3NpdG9yeToge3NlbGVjdGlvbi5yZXBvX2lkfSIpCiAgICAgICAgcHJpbnQoZiJRdWFudDoge3NlbGVjdGlvbi5xdWFudH0iKQogICAgICAgIHByaW50KGYiVG90YWwgbW9kZWwgd2VpZ2h0IHNpemU6IHtzZWxlY3Rpb24udG90YWxfc2l6ZV9ieXRlcyAvIEdJQjouMmZ9IEdpQiIpCiAgICAgICAgcHJpbnQoZiJQcmltYXJ5IEdHVUY6IHtzZWxlY3Rpb24ucHJpbWFyeV9maWxlfSIpCiAgICAgICAgcHJpbnQoIkZpbGVzIHRvIGRvd25sb2FkOiIpCiAgICAgICAgZm9yIGZpbGVuYW1lIGluIHNlbGVjdGlvbi5maWxlczoKICAgICAgICAgICAgcHJpbnQoZiIgIHtmaWxlbmFtZX0iKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg=="
}
for filename, content in embedded_files.items():
    file_path = server_dir / filename
    file_path.write_bytes(base64.b64decode(content))
    if filename.endswith('.sh'):
        file_path.chmod(0o700)
print(f"Prepared server scripts in {server_dir}")


In [ ]:
%cd /kaggle/working/qwenfree_server
!bash bootstrap.sh
